In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import glob
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

# Spremenjen encoding standard iz utf8 na cp1250
testni_podatki = pd.read_csv(
    "podatki/ovrednoteni_podatki/m0.csv", 
    sep=",",           # Changed from ";" to ","
    decimal=".",       # Changed from "," to "."
    header=None,       # Added because the first line is data, not column names
    encoding="cp1250"
)

#df = pd.concat([df2023, df2022])
#df = df.reset_index(drop=True)

testni_podatki.columns
sirina,visina = testni_podatki.shape
sirina
testni_podatki.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'podatki/ovrednoteni_podatki/m0.csv'

In [2]:

# 1. Get a list of all CSV filenames in the folder
# Assuming your files are named like m0.csv, m1.csv, etc.
path = 'podatki/ovrednoteni_podatki/' # Use the path to your folder
all_files = glob.glob(path + "m*.csv") 

# 2. Use a list comprehension to read all files at once
# We include the same settings we found for your 'm0.csv'
li = []
for filename in all_files:
    df_temp = pd.read_csv(filename, sep=",", decimal=".", header=None, encoding="cp1250")
    # Optional: Add a column to keep track of which file the data came from
    #df_temp['source_file'] = filename 
    li.append(df_temp)

# 3. Concatenate everything into one big DataFrame
df_oznaceno = pd.concat(li, axis=0, ignore_index=True)

# 4. Name the columns (using the names we discussed)
df_oznaceno.columns = ["ID Merilnika", "Čas (DD.MM HH)", "meritev", "anomalija"]

df_oznaceno.head(-5)

In [3]:
df_oznaceno["anomalija"].value_counts()

In [4]:
# Sort by ID and Time first!
df_oznaceno = df_oznaceno.sort_values(['ID Merilnika', 'Čas (DD.MM HH)'])

# Create features that look at the last 3 time steps
for i in range(1, 4):
    df_oznaceno[f'lag_{i}'] = df_oznaceno.groupby('ID Merilnika')['meritev'].shift(i)

# Create rolling stats (Standard Deviation is great for spotting the start of noise)
df_oznaceno['rolling_mean'] = df_oznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).mean())
df_oznaceno['rolling_std'] = df_oznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).std())

# How much did the measurement change since the last step?
df_oznaceno['velocity'] = df_oznaceno.groupby('ID Merilnika')['meritev'].diff()

# How much is the current value deviating from the local average?
#df_oznaceno['dev_from_mean'] = df_oznaceno['meritev'] - df_oznaceno['rolling_mean']

In [5]:
# 1. Create df_clean by dropping the NaN rows created by lags/rolling windows
df_clean = df_oznaceno.dropna().copy()

# 2. Make sure your features list includes your new 'velocity' column
features = ['meritev', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean', 'rolling_std', 'velocity']

# 3. Now you can safely sort and split
df_clean = df_clean.sort_values('Čas (DD.MM HH)')

In [6]:
# 1. Ensure data is sorted by time globally (this is crucial!)
df_clean = df_clean.sort_values('Čas (DD.MM HH)')

X = df_clean[features]
y = df_clean['anomalija']

# 2. Initialize TimeSeriesSplit (n_splits=5 is standard)
tscv = TimeSeriesSplit(n_splits=5)

# 3. Iterate through the folds
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Train the model on this specific fold
    model = xgb.XGBClassifier(
        #scale_pos_weight=11.1, 
        random_state=42)
    model.fit(X_train, y_train)
    
    # Use a lower threshold to 'catch' more anomalies
    probs = model.predict_proba(X_test)[:, 1]
    predictions = (probs > 0.35).astype(int)
    print(f"Fold Results:\n{classification_report(y_test, predictions)}")

In [7]:
# 1. Ensure we have the final predictions (using your tuned threshold)
probs = model.predict_proba(X)[:, 1]
df_clean['pred_anom'] = (probs > 0.35).astype(int)

# 2. Identify the transitions (grouped by ID so houses don't bleed into each other)
df_clean['transition'] = df_clean.groupby('ID Merilnika')['pred_anom'].diff()

# 3. Extract the Timestamps
interruption_list = []

for house_id, group in df_clean.groupby('ID Merilnika'):
    # Get timestamps for all starts (1) and all ends (-1)
    starts = group[group['transition'] == 1]['Čas (DD.MM HH)'].tolist()
    ends = group[group['transition'] == -1]['Čas (DD.MM HH)'].tolist()
    
    # Match them up into pairs
    # Note: min() handles cases where an anomaly might still be ongoing at the end of the file
    for i in range(min(len(starts), len(ends))):
        interruption_list.append({
            'Meter_ID': house_id,
            'Start_Time': starts[i],
            'End_Time': ends[i]
        })

# 4. Create the final report
df_report = pd.DataFrame(interruption_list)

# FIX: Put the year immediately after the date, or keep it at the end but fix the format string
# Let's change the string concatenation to: "10.03.2026 15:00"
df_report['Start_Time'] = pd.to_datetime(
    df_report['Start_Time'].str.replace(' ', '.2026 '), 
    format='%d.%m.%Y %H:%M'
)
df_report['End_Time'] = pd.to_datetime(
    df_report['End_Time'].str.replace(' ', '.2026 '), 
    format='%d.%m.%Y %H:%M'
)

# Calculate Duration
df_report['Duration'] = df_report['End_Time'] - df_report['Start_Time']

# Sort and display
df_report = df_report.sort_values(by='Duration', ascending=False)
print(df_report.head(10))

In [8]:

# 1. Get a list of all CSV filenames in the folder
# Assuming your files are named like m0.csv, m1.csv, etc.
path = 'podatki/vsi_podatki/' # Use the path to your folder
all_files = glob.glob(path + "m*.csv") 

# 2. Use a list comprehension to read all files at once
# We include the same settings we found for your 'm0.csv'
li = []
for filename in all_files:
    df_temp = pd.read_csv(filename, sep=",", decimal=".", header=None, encoding="cp1250")
    # Optional: Add a column to keep track of which file the data came from
    #df_temp['source_file'] = filename 
    li.append(df_temp)

# 3. Concatenate everything into one big DataFrame
df_neoznaceno = pd.concat(li, axis=0, ignore_index=True)

# 4. Name the columns (using the names we discussed)
df_neoznaceno.columns = ["ID Merilnika", "Čas (DD.MM HH)", "meritev"]

df_neoznaceno.head(-5)

In [9]:
# 1. Sort by ID and Time to ensure correct calculations
df_neoznaceno = df_neoznaceno.sort_values(['ID Merilnika', 'Čas (DD.MM HH)'])

# 2. Re-create the features (Must match the training features list exactly)
for i in range(1, 4):
    df_neoznaceno[f'lag_{i}'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].shift(i)

df_neoznaceno['rolling_mean'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).mean())
df_neoznaceno['rolling_std'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).std())
df_neoznaceno['velocity'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].diff()

# 3. Handle NaNs created by lags/rolling windows
df_predict_clean = df_neoznaceno.dropna().copy()

# 4. Define the feature matrix for prediction
X_new = df_predict_clean[features]

In [10]:
# 1. Get probabilities and apply the threshold
# We use the model trained in the previous cells
probs_new = model.predict_proba(X_new)[:, 1]
df_predict_clean['pred_anom'] = (probs_new > 0.35).astype(int)

print(f"Total rows in new data: {len(df_predict_clean)}")
print(f"Anomalous points detected: {df_predict_clean['pred_anom'].sum()}")

In [1]:
# 1. Identify transitions (1 = Start, -1 = End)
df_predict_clean['transition'] = df_predict_clean.groupby('ID Merilnika')['pred_anom'].diff()

final_anomalies = []

for house_id, group in df_predict_clean.groupby('ID Merilnika'):
    starts = group[group['transition'] == 1]['Čas (DD.MM HH)'].tolist()
    ends = group[group['transition'] == -1]['Čas (DD.MM HH)'].tolist()
    
    # Pair up starts and ends
    for i in range(min(len(starts), len(ends))):
        final_anomalies.append({
            'Meter_ID': house_id,
            'Start_Time': starts[i],
            'End_Time': ends[i]
        })

# 2. Convert to DataFrame and format
df_submission = pd.DataFrame(final_anomalies)

if not df_submission.empty:
    # Adding the year and converting to datetime for duration math
    df_submission['Start_Time_dt'] = pd.to_datetime(df_submission['Start_Time'].str.replace(' ', '.2026 '), format='%d.%m.%Y %H:%M')
    df_submission['End_Time_dt'] = pd.to_datetime(df_submission['End_Time'].str.replace(' ', '.2026 '), format='%d.%m.%Y %H:%M')
    
    df_submission['Duration'] = df_submission['End_Time_dt'] - df_submission['Start_Time_dt']
    
    # Final cleanup for submission
    df_submission = df_submission.sort_values(['Meter_ID', 'Start_Time_dt'])
    print("Top 10 Detected Anomalies:")
    display(df_submission[['Meter_ID', 'Start_Time', 'End_Time', 'Duration']].head(10))
else:
    print("No anomalies found.")

NameError: name 'df_predict_clean' is not defined

In [12]:
# Save the results to a CSV file
df_submission[['Meter_ID', 'Start_Time', 'End_Time', 'Duration']].to_csv('final_predictions.csv', index=False)
print("Results saved to final_predictions.csv")